# Improved Unistroke Drawing Recognition
Here we normalize the data, apply fixed-size resampling, compute local velocity (dx, dy), and train a robust Bidirectional LSTM.

In [ ]:
import json
import numpy as np
import string
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
from sklearn.model_selection import train_test_split


In [ ]:
print("Loading and normalising data...")
with open('xbox_draw/stroke_data.json', 'r') as f:
    dataset = json.load(f)

num_points = 50
char_to_int = {c: i for i, c in enumerate(string.ascii_lowercase)}

X_list = []
y_list = []

def process_stroke(stroke):
    if len(stroke) < 2: return None
    x = np.array([pt['x'] for pt in stroke])
    y = np.array([pt['y'] for pt in stroke])
    
    # Resample to exactly num_points
    old_indices = np.linspace(0, 1, len(stroke))
    new_indices = np.linspace(0, 1, num_points)
    new_x = np.interp(new_indices, old_indices, x)
    new_y = np.interp(new_indices, old_indices, y)
    
    # Normalize: zero mean, bounded scale
    new_x -= np.mean(new_x)
    new_y -= np.mean(new_y)
    scale = max(np.max(np.abs(new_x)), np.max(np.abs(new_y))) + 1e-8
    new_x /= scale
    new_y /= scale
    
    # Compute derivatives (dx, dy) as extra features
    dx = np.diff(new_x, prepend=new_x[0])
    dy = np.diff(new_y, prepend=new_y[0])
    
    return np.column_stack([new_x, new_y, dx, dy])

for item in dataset:
    label = item.get('label')
    seq = item.get('sequence', [])
    if label not in char_to_int: 
        continue
    
    features = process_stroke(seq)
    if features is not None:
        X_list.append(features)
        y_list.append(char_to_int[label])

X = np.array(X_list, dtype='float32')
y = np.array(y_list)

print("Data shapes:", X.shape, y.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print("Building improved model...")
num_classes = 26
input_shape = (num_points, 4) # x, y, dx, dy

model = Sequential([
    Bidirectional(LSTM(128, return_sequences=True), input_shape=input_shape),
    Dropout(0.3),
    Bidirectional(LSTM(64)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
print("Training model...")
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(X_train, y_train, epochs=80, batch_size=32, validation_data=(X_test, y_test), callbacks=[early_stop])

In [ ]:
print("Saving model...")
model.save('stroke_model.h5')
print("Model saved to stroke_model.h5.")